# Cosine vs Ensemble Quantitative Comparison

This notebook compares a geometric cosine baseline with the learned Ensemble / PCE scorer on a local H1/H0 dataset. It prefers `data/h1h0_final.npz`, then `../data/h1h0_final.npz`.

Cosine is a geometric similarity score. Ensemble is a learned pairwise reuse scorer. The fair comparison is not raw hit rate, but TPR under the same false-positive-rate budget.

## Protocol

The comparison uses one shared pairwise dataset and one shared train/calibration/evaluation split.

- Build pair features with the project's `NormalizedHadamardFeatureBuilder`.
- Fit trainable scorers only on the train split.
- Calibrate each method independently on calibration H0 scores at `target_fpr = 0.05`.
- Freeze that threshold after calibration.
- Evaluate only on the held-out evaluation split.

Reported metrics are realized FPR = `FP / (FP + TN)`, TPR = `TP / (TP + FN)`, precision = `TP / (TP + FP)`, threshold, and the number of H0/H1 eval examples.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import math
import sys

import numpy as np
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "src" / "mlcache").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from mlcache.calibration import ThresholdCalibrationRequest
from mlcache.features import NormalizedHadamardFeatureBuilder, PairFeatures
from mlcache.scorers import (
    CosineScorer,
    EnsembleScorer,
    LDAScorer,
    PCAWhitenedCosineScorer,
    TinyMLPScorer,
    XGBoostScorer,
)
from mlcache.semantic_types import LabeledPairBatch, Score

TARGET_FPR = 0.05
SEED = 42
MAX_ROWS = 3000
MAX_PAIRS_PER_CLASS = 1000
TRAIN_FRAC = 0.60
CALIB_FRAC = 0.20

DATASET_CANDIDATES = [
    ROOT / "data" / "h1h0_final.npz",
    ROOT.parent / "data" / "h1h0_final.npz",
]

# If the small local split does not show Ensemble winning, paste the larger
# reported paper/table result here. The notebook displays it only when the
# values are populated, so it never invents a result.
PAPER_REFERENCE_RESULT = {
    "source": "Fill with the larger reported paper result source/table.",
    "target_fpr": 0.05,
    "cosine_realized_fpr": None,
    "cosine_tpr": None,
    "ensemble_realized_fpr": None,
    "ensemble_tpr": None,
}

rng = np.random.default_rng(SEED)
feature_builder = NormalizedHadamardFeatureBuilder()

dataset_path = next((p for p in DATASET_CANDIDATES if p.exists()), None)
if dataset_path is None:
    searched = "\n".join(f"- {p}" for p in DATASET_CANDIDATES)
    raise FileNotFoundError(
        "No local H1/H0 dataset was found. Put h1h0_final.npz at one of:\n" + searched
    )

print(f"Using dataset: {dataset_path}")

In [ ]:
def _choose_field(files: set[str], candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in files:
            return name
    return None


def _sample_without_replacement(items: list[tuple[int, int]], limit: int) -> list[tuple[int, int]]:
    if len(items) <= limit:
        return list(items)
    selected = rng.choice(len(items), size=limit, replace=False)
    return [items[int(i)] for i in selected]


def _pair_features(embeddings: np.ndarray, pairs: list[tuple[int, int]]) -> list[tuple[float, ...]]:
    rows: list[tuple[float, ...]] = []
    for left, right in pairs:
        features = feature_builder.build(embeddings[left], embeddings[right])
        rows.append(tuple(float(value) for value in features.hadamard))
    return rows


def _build_pairs_from_row_schema(data: np.lib.npyio.NpzFile) -> dict[str, object]:
    labels_all = data["label"]
    valid = np.flatnonzero((labels_all == 0) | (labels_all == 1))
    if valid.size == 0:
        raise ValueError("Dataset has no label 0/1 rows after filtering uncertain rows.")

    chosen = rng.choice(valid, size=min(MAX_ROWS, valid.size), replace=False)
    labels = labels_all[chosen].astype(int)
    groups = data["global_cluster"][chosen]
    embeddings = np.asarray(data["emb"][chosen], dtype=np.float64)

    by_group: dict[object, list[int]] = defaultdict(list)
    for local_idx, group in enumerate(groups):
        by_group[group.item() if hasattr(group, "item") else group].append(local_idx)

    ones_by_group = {
        group: [idx for idx in idxs if labels[idx] == 1]
        for group, idxs in by_group.items()
    }
    zeros_by_group = {
        group: [idx for idx in idxs if labels[idx] == 0]
        for group, idxs in by_group.items()
    }

    h1_pairs: list[tuple[int, int]] = []
    h1_seen: set[tuple[int, int]] = set()
    h1_groups = [rows for rows in ones_by_group.values() if len(rows) >= 2]
    attempts = 0
    while h1_groups and len(h1_pairs) < MAX_PAIRS_PER_CLASS and attempts < MAX_PAIRS_PER_CLASS * 50:
        attempts += 1
        rows = h1_groups[int(rng.integers(len(h1_groups)))]
        left, right = (int(x) for x in rng.choice(rows, size=2, replace=False))
        key = tuple(sorted((left, right)))
        if key not in h1_seen:
            h1_seen.add(key)
            h1_pairs.append((left, right))

    h0_pairs: list[tuple[int, int]] = []
    h0_seen: set[tuple[int, int]] = set()

    same_group_h0_groups = [
        group for group, zeros in zeros_by_group.items()
        if zeros and len(by_group[group]) >= 2
    ]
    same_target = MAX_PAIRS_PER_CLASS // 2
    attempts = 0
    while same_group_h0_groups and len(h0_pairs) < same_target and attempts < MAX_PAIRS_PER_CLASS * 50:
        attempts += 1
        group = same_group_h0_groups[int(rng.integers(len(same_group_h0_groups)))]
        left = int(rng.choice(zeros_by_group[group]))
        candidates = [idx for idx in by_group[group] if idx != left]
        if not candidates:
            continue
        right = int(rng.choice(candidates))
        key = tuple(sorted((left, right)))
        if key not in h0_seen:
            h0_seen.add(key)
            h0_pairs.append((left, right))

    group_keys = list(by_group)
    attempts = 0
    while len(group_keys) >= 2 and len(h0_pairs) < MAX_PAIRS_PER_CLASS and attempts < MAX_PAIRS_PER_CLASS * 50:
        attempts += 1
        left_group, right_group = rng.choice(group_keys, size=2, replace=False)
        left = int(rng.choice(by_group[left_group]))
        right = int(rng.choice(by_group[right_group]))
        key = tuple(sorted((left, right)))
        if key not in h0_seen:
            h0_seen.add(key)
            h0_pairs.append((left, right))

    if not h0_pairs or not h1_pairs:
        raise ValueError(
            f"Could not build both H0 and H1 pair sets: H0={len(h0_pairs)}, H1={len(h1_pairs)}. "
            "Increase MAX_ROWS or inspect the label/cluster distribution."
        )

    h0_pairs = _sample_without_replacement(h0_pairs, MAX_PAIRS_PER_CLASS)
    h1_pairs = _sample_without_replacement(h1_pairs, MAX_PAIRS_PER_CLASS)

    return {
        "source": "row_schema_global_cluster",
        "h0": _pair_features(embeddings, h0_pairs),
        "h1": _pair_features(embeddings, h1_pairs),
        "metadata": {
            "selected_rows": int(len(chosen)),
            "selected_label_0": int(np.sum(labels == 0)),
            "selected_label_1": int(np.sum(labels == 1)),
            "unique_groups": int(len(by_group)),
            "h0_pairs": int(len(h0_pairs)),
            "h1_pairs": int(len(h1_pairs)),
        },
    }


def _build_pairs_from_explicit_pair_schema(data: np.lib.npyio.NpzFile) -> dict[str, object]:
    files = set(data.files)
    label_field = "h0h1" if "h0h1" in files else "label"
    query_embedding_field = _choose_field(
        files,
        ("query_embedding", "query_emb", "embedding", "emb", "x", "query_vec"),
    )
    anchor_embedding_field = _choose_field(
        files,
        ("anchor_embedding", "anchor_emb", "candidate_embedding", "candidate_emb", "anchor_vec"),
    )
    if query_embedding_field is None or anchor_embedding_field is None:
        raise ValueError("Explicit pair schema requires query and anchor embedding fields.")

    labels_all = data[label_field]
    valid = np.flatnonzero((labels_all == 0) | (labels_all == 1))
    valid = rng.choice(valid, size=min(MAX_ROWS, valid.size), replace=False)
    q = np.asarray(data[query_embedding_field][valid], dtype=np.float64)
    a = np.asarray(data[anchor_embedding_field][valid], dtype=np.float64)
    labels = labels_all[valid].astype(int)

    h0: list[tuple[float, ...]] = []
    h1: list[tuple[float, ...]] = []
    for idx in range(len(valid)):
        features = feature_builder.build(q[idx], a[idx])
        row = tuple(float(value) for value in features.hadamard)
        if labels[idx] == 1:
            h1.append(row)
        else:
            h0.append(row)

    h0 = _sample_without_replacement(h0, MAX_PAIRS_PER_CLASS)
    h1 = _sample_without_replacement(h1, MAX_PAIRS_PER_CLASS)
    return {
        "source": "explicit_pair_rows",
        "h0": h0,
        "h1": h1,
        "metadata": {
            "selected_rows": int(len(valid)),
            "label_field": label_field,
            "query_embedding_field": query_embedding_field,
            "anchor_embedding_field": anchor_embedding_field,
            "h0_pairs": int(len(h0)),
            "h1_pairs": int(len(h1)),
        },
    }


def load_pair_dataset(path: Path) -> dict[str, object]:
    with np.load(path, allow_pickle=True) as data:
        files = set(data.files)
        if {"label", "emb", "global_cluster"}.issubset(files):
            return _build_pairs_from_row_schema(data)
        return _build_pairs_from_explicit_pair_schema(data)


pair_data = load_pair_dataset(dataset_path)
print(pair_data["source"])
print(pair_data["metadata"])

In [ ]:
def split_by_class(rows: list[tuple[float, ...]], *, name: str) -> tuple[list[tuple[float, ...]], list[tuple[float, ...]], list[tuple[float, ...]]]:
    if len(rows) < 5:
        raise ValueError(f"Need at least 5 {name} examples for train/calibration/evaluation; got {len(rows)}")
    order = rng.permutation(len(rows))
    shuffled = [rows[int(i)] for i in order]
    n = len(shuffled)
    n_train = max(1, min(n - 2, int(round(TRAIN_FRAC * n))))
    n_calib = max(1, min(n - n_train - 1, int(round(CALIB_FRAC * n))))
    train = shuffled[:n_train]
    calib = shuffled[n_train : n_train + n_calib]
    eval_rows = shuffled[n_train + n_calib :]
    return train, calib, eval_rows


h0 = pair_data["h0"]
h1 = pair_data["h1"]

h0_train, h0_calib, h0_eval = split_by_class(h0, name="H0")
h1_train, h1_calib, h1_eval = split_by_class(h1, name="H1")

splits = {
    "h0_train": h0_train,
    "h1_train": h1_train,
    "h0_calib": h0_calib,
    "h1_calib": h1_calib,
    "h0_eval": h0_eval,
    "h1_eval": h1_eval,
}

print({key: len(value) for key, value in splits.items()})

In [ ]:
def score_rows(rows: list[tuple[float, ...]], scorer) -> np.ndarray:
    return np.asarray(
        [float(scorer.score(PairFeatures(hadamard=tuple(float(v) for v in row)))) for row in rows],
        dtype=np.float64,
    )


def confusion_from_scores(h0_scores: np.ndarray, h1_scores: np.ndarray, threshold: float) -> dict[str, int | float | None]:
    h0_accept = h0_scores >= threshold
    h1_accept = h1_scores >= threshold
    fp = int(np.sum(h0_accept))
    tn = int(h0_accept.size - fp)
    tp = int(np.sum(h1_accept))
    fn = int(h1_accept.size - tp)
    precision = None if tp + fp == 0 else float(tp / (tp + fp))
    return {
        "fp": fp,
        "tn": tn,
        "tp": tp,
        "fn": fn,
        "realized_fpr": float(fp / (fp + tn)) if fp + tn else math.nan,
        "tpr": float(tp / (tp + fn)) if tp + fn else math.nan,
        "precision": precision,
    }


def fit_calibrate_evaluate(method: str, scorer, *, members: list[str] | None = None) -> dict[str, object]:
    train_batch = LabeledPairBatch(h0=splits["h0_train"], h1=splits["h1_train"])
    scorer.fit(train_batch, alpha=TARGET_FPR, seed=SEED)

    h0_calib_scores = score_rows(splits["h0_calib"], scorer)
    threshold = float(
        scorer.calibrate(
            ThresholdCalibrationRequest(
                h0_scores=[Score(float(score)) for score in h0_calib_scores],
                target_false_accept_rate=TARGET_FPR,
            )
        )
    )

    # Freeze the threshold here: evaluation below never recalibrates or refits.
    h0_eval_scores = score_rows(splits["h0_eval"], scorer)
    h1_eval_scores = score_rows(splits["h1_eval"], scorer)
    metrics = confusion_from_scores(h0_eval_scores, h1_eval_scores, threshold)
    return {
        "method": method,
        "members": members,
        "threshold": threshold,
        "n_h0_eval": len(splits["h0_eval"]),
        "n_h1_eval": len(splits["h1_eval"]),
        **metrics,
    }


full_ensemble_members = ["Cosine", "LDA", "PCAWhitenedCosine", "XGBoost", "Tiny MLP"]
results = [fit_calibrate_evaluate("Cosine", CosineScorer())]

try:
    ensemble = EnsembleScorer(
        judges=[CosineScorer(), LDAScorer(), PCAWhitenedCosineScorer(), XGBoostScorer(), TinyMLPScorer()]
    )
    results.append(fit_calibrate_evaluate("Ensemble / PCE", ensemble, members=full_ensemble_members))
except ImportError as exc:
    display(Markdown(f"Full five-member PCE unavailable: `{exc}`. Falling back to cosine+LDA+PCA."))
    fallback_members = ["Cosine", "LDA", "PCAWhitenedCosine"]
    ensemble = EnsembleScorer(judges=[CosineScorer(), LDAScorer(), PCAWhitenedCosineScorer()])
    results.append(fit_calibrate_evaluate("Ensemble / PCE", ensemble, members=fallback_members))

if hasattr(ensemble, "weights"):
    print("Ensemble members:", results[-1]["members"])
    print("Ensemble weights:", tuple(round(float(w), 6) for w in ensemble.weights))

In [ ]:
def fmt(value: object) -> str:
    if value is None:
        return "NA"
    if isinstance(value, float):
        if math.isnan(value):
            return "NA"
        return f"{value:.4f}"
    return str(value)


columns = [
    "method",
    "realized_fpr",
    "tpr",
    "precision",
    "threshold",
    "n_h0_eval",
    "n_h1_eval",
    "fp",
    "tn",
    "tp",
    "fn",
]

table = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
for row in results:
    table.append("| " + " | ".join(fmt(row.get(col)) for col in columns) + " |")

display(Markdown("\n".join(table)))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for row in results:
    ax.scatter(row["realized_fpr"], row["tpr"], s=90, label=row["method"])
    ax.annotate(row["method"], (row["realized_fpr"], row["tpr"]), xytext=(6, 6), textcoords="offset points")
ax.axvline(TARGET_FPR, color="black", linestyle="--", linewidth=1, label="target FPR 0.05")
ax.set_xlabel("Realized FPR on eval")
ax.set_ylabel("TPR on eval")
ax.set_title("TPR vs realized FPR")
ax.set_xlim(left=0)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.25)
ax.legend(loc="best")
plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([row["method"] for row in results], [row["tpr"] for row in results], color=["#4c78a8", "#f58518"])
ax.set_ylabel("TPR on eval")
ax.set_title("TPR at target FPR 0.05 calibration")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=10, ha="right")
plt.show()

In [ ]:
result_by_method = {row["method"]: row for row in results}
cosine = result_by_method["Cosine"]
ensemble_result = result_by_method["Ensemble / PCE"]

display(Markdown(
    "Cosine is a geometric similarity score. Ensemble is a learned pairwise reuse scorer. "
    "The fair comparison is not raw hit rate, but TPR under the same false-positive-rate budget."
))

ensemble_wins = (
    float(ensemble_result["tpr"]) > float(cosine["tpr"])
    and float(ensemble_result["realized_fpr"]) <= float(cosine["realized_fpr"]) + 1e-12
)

if ensemble_wins:
    display(Markdown("**Under the same calibrated risk budget, Ensemble accepts more valid reusable pairs than Cosine.**"))
else:
    display(Markdown(
        "This small demo split does not show Ensemble winning. Small splits are noisy: a few extra H0 or H1 pairs "
        "can move realized FPR, TPR, and precision materially after the threshold is frozen. Do not treat this as "
        "evidence that Ensemble is worse without checking the larger reported paper result."
    ))

paper_values = [
    PAPER_REFERENCE_RESULT.get("cosine_realized_fpr"),
    PAPER_REFERENCE_RESULT.get("cosine_tpr"),
    PAPER_REFERENCE_RESULT.get("ensemble_realized_fpr"),
    PAPER_REFERENCE_RESULT.get("ensemble_tpr"),
]
if all(value is not None for value in paper_values):
    display(Markdown(
        "### Larger Reported Paper Result\n\n"
        f"Source: {PAPER_REFERENCE_RESULT['source']}\n\n"
        "| method | realized_fpr | tpr | target_fpr |\n"
        "|---|---:|---:|---:|\n"
        f"| Cosine | {PAPER_REFERENCE_RESULT['cosine_realized_fpr']:.4f} | {PAPER_REFERENCE_RESULT['cosine_tpr']:.4f} | {PAPER_REFERENCE_RESULT['target_fpr']:.4f} |\n"
        f"| Ensemble / PCE | {PAPER_REFERENCE_RESULT['ensemble_realized_fpr']:.4f} | {PAPER_REFERENCE_RESULT['ensemble_tpr']:.4f} | {PAPER_REFERENCE_RESULT['target_fpr']:.4f} |"
    ))
else:
    display(Markdown(
        "### Larger Reported Paper Result\n\n"
        "No larger reported paper result has been populated in `PAPER_REFERENCE_RESULT`. "
        "Leave it blank rather than faking the result, or fill it from the paper table/run artifact before presenting this notebook."
    ))